# Final semantic CSV analysis

최종 산출물인 `stay_manual_semantic_clean.csv`, `visa_manual_semantic_clean.csv`만 읽어 구조, 분포, 누락률, 주요 코드/항목을 확인합니다.

이 노트북은 `matplotlib` 기반 시각화를 사용합니다. PDF page, evidence, raw, review 컬럼은 최종 CSV에 없다는 전제를 검증합니다.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'processed').exists():
    ROOT = (Path.cwd() / '..').resolve()
PROCESSED = ROOT / 'data' / 'processed'
FIG_DIR = ROOT / 'output' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

stay_path = PROCESSED / 'stay_manual_semantic_clean.csv'
visa_path = PROCESSED / 'visa_manual_semantic_clean.csv'

stay = pd.read_csv(stay_path, dtype=str).fillna('')
visa = pd.read_csv(visa_path, dtype=str).fillna('')

font_path = Path('/System/Library/Fonts/Supplemental/AppleGothic.ttf')
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = fm.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'

print(stay_path, stay.shape)
print(visa_path, visa.shape)

## 1. 최종 CSV 계약 확인

금지 컬럼이 없는지, `data/processed`에 최종 CSV 2개만 남았는지 확인합니다.

In [ ]:
forbidden_tokens = ['page', 'raw', 'evidence', 'review', 'section_path']
processed_files = sorted(p.name for p in PROCESSED.iterdir() if p.is_file())

contract = []
for name, df in [('체류민원', stay), ('사증민원', visa)]:
    forbidden_cols = [c for c in df.columns if any(token in c for token in forbidden_tokens)]
    contract.append({
        'manual': name,
        'rows': len(df),
        'columns': len(df.columns),
        'forbidden_columns': ', '.join(forbidden_cols),
        'noise_title_rows': int(df['section_title'].isin(['目 次', '목차', '▶ 목차', '▣ 목차']).sum()),
        'document_rows': int((df[['common_documents', 'mandatory_documents', 'other_documents']].apply(lambda s: s.str.strip() != '').any(axis=1)).sum()),
    })

display(pd.DataFrame(contract))
print('processed files:', processed_files)

## 2. 분석용 통합 데이터프레임

In [ ]:
stay_plot = stay.copy()
stay_plot['manual'] = '체류민원'
stay_plot['code'] = stay_plot['stay_status_code'].replace('', '공통/무코드')

visa_plot = visa.copy()
visa_plot['manual'] = '사증민원'
visa_plot['code'] = visa_plot['visa_code'].replace('', '공통/무코드')

common_analysis_cols = [
    'manual', 'item_type', 'subsection_type', 'petition_type', 'code', 'section_title',
    'common_documents', 'mandatory_documents', 'other_documents',
    'eligibility', 'target_persons', 'requirements', 'procedure',
    'restrictions', 'exceptions', 'fees', 'duration_or_validity',
    'quota_or_limit', 'score_criteria', 'table_rows', 'normalized_text',
]
combined = pd.concat([stay_plot[common_analysis_cols], visa_plot[common_analysis_cols]], ignore_index=True)
combined.head()

## 3. Matplotlib 대시보드

row 수, item type, subsection type, 주요 코드, 서류 필드, 주요 필드 채움률을 한 번에 봅니다.

In [ ]:
colors = ['#2f6f73', '#b36b32']
fig, axes = plt.subplots(3, 2, figsize=(18, 18), constrained_layout=True)
fig.suptitle('최종 Semantic CSV 현황', fontsize=22, fontweight='bold')

manual_counts = combined['manual'].value_counts().reindex(['체류민원', '사증민원'])
ax = axes[0, 0]
bars = ax.bar(manual_counts.index, manual_counts.values, color=colors)
ax.set_title('매뉴얼별 row 수', fontsize=15, fontweight='bold')
ax.set_ylabel('rows')
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{int(b.get_height()):,}', ha='center', va='bottom', fontsize=12)

ax = axes[0, 1]
item_counts = combined.pivot_table(index='item_type', columns='manual', aggfunc='size', fill_value=0)
item_counts['total'] = item_counts.sum(axis=1)
item_counts = item_counts.sort_values('total', ascending=True).drop(columns='total').tail(10)
item_counts.plot(kind='barh', ax=ax, color=colors)
ax.set_title('item_type 분포 Top 10', fontsize=15, fontweight='bold')
ax.set_xlabel('rows')
ax.set_ylabel('')
ax.legend(title='')

ax = axes[1, 0]
sub_counts = combined.pivot_table(index='subsection_type', columns='manual', aggfunc='size', fill_value=0)
sub_counts['total'] = sub_counts.sum(axis=1)
sub_counts = sub_counts.sort_values('total', ascending=True).drop(columns='total').tail(12)
sub_counts.plot(kind='barh', ax=ax, color=colors)
ax.set_title('subsection_type 분포 Top 12', fontsize=15, fontweight='bold')
ax.set_xlabel('rows')
ax.set_ylabel('')
ax.legend(title='')

ax = axes[1, 1]
code_counts = combined[combined['code'] != '공통/무코드']['code'].value_counts().head(15).sort_values()
ax.barh(code_counts.index, code_counts.values, color='#4d79a8')
ax.set_title('코드별 row 수 Top 15', fontsize=15, fontweight='bold')
ax.set_xlabel('rows')
ax.set_ylabel('')
for y, v in enumerate(code_counts.values):
    ax.text(v, y, f' {int(v):,}', va='center', fontsize=10)

ax = axes[2, 0]
doc_cols = ['common_documents', 'mandatory_documents', 'other_documents']
doc_labels = ['공통서류', '필수서류', '기타서류']
doc_data = pd.DataFrame({
    '체류민원': [(stay[c].str.strip() != '').sum() for c in doc_cols],
    '사증민원': [(visa[c].str.strip() != '').sum() for c in doc_cols],
}, index=doc_labels)
doc_data.plot(kind='bar', ax=ax, color=colors, rot=0)
ax.set_title('서류 필드가 채워진 row 수', fontsize=15, fontweight='bold')
ax.set_ylabel('rows')
ax.legend(title='')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=10)

ax = axes[2, 1]
fields = ['eligibility', 'requirements', 'procedure', 'restrictions', 'exceptions', 'fees', 'duration_or_validity', 'quota_or_limit', 'score_criteria', 'table_rows']
field_labels = ['대상', '요건', '절차', '제한', '예외', '수수료', '기간/유효', '쿼터', '점수', '표행']
rate = pd.DataFrame({
    '체류민원': [(stay[f].str.strip() != '').mean() * 100 for f in fields],
    '사증민원': [(visa[f].str.strip() != '').mean() * 100 for f in fields],
}, index=field_labels)
rate.plot(kind='bar', ax=ax, color=colors, rot=35)
ax.set_title('주요 필드 채움률', fontsize=15, fontweight='bold')
ax.set_ylabel('% of rows')
ax.set_ylim(0, max(10, rate.to_numpy().max() * 1.25))
ax.legend(title='')

for ax in axes.ravel():
    ax.grid(axis='x', alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)

fig.text(0.5, 0.01, f'체류민원 {len(stay):,} rows / 사증민원 {len(visa):,} rows', ha='center', fontsize=12, color='#333333')
plt.show()

## 4. Subsection과 item_type 교차 분석

In [ ]:
item_subsection = pd.crosstab(combined['subsection_type'], combined['item_type'])
item_subsection.loc[item_subsection.sum(axis=1).sort_values(ascending=False).index].head(15)

## 5. 민원 유형 분포

In [ ]:
petition_counts = combined['petition_type'].replace('', '미분류').value_counts().head(20)
fig, ax = plt.subplots(figsize=(12, 7))
petition_counts.sort_values().plot(kind='barh', ax=ax, color='#6d7f3f')
ax.set_title('petition_type 분포 Top 20', fontsize=15, fontweight='bold')
ax.set_xlabel('rows')
ax.grid(axis='x', alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)
plt.show()
petition_counts.to_frame('rows')

## 6. 주요 코드별 subsection 분포

많이 등장하는 코드가 어떤 항목으로 구성되어 있는지 봅니다.

In [ ]:
top_codes = combined.loc[combined['code'] != '공통/무코드', 'code'].value_counts().head(10).index
code_subsection = pd.crosstab(combined.loc[combined['code'].isin(top_codes), 'code'], combined.loc[combined['code'].isin(top_codes), 'subsection_type'])
code_subsection = code_subsection.loc[top_codes]

fig, ax = plt.subplots(figsize=(14, 7))
code_subsection.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_title('주요 코드별 subsection_type 구성', fontsize=15, fontweight='bold')
ax.set_ylabel('rows')
ax.set_xlabel('code')
ax.legend(title='subsection_type', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(axis='y', alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)
plt.show()
code_subsection

## 7. 누락률/채움률 테이블

In [ ]:
analysis_fields = ['common_documents', 'mandatory_documents', 'other_documents', 'eligibility', 'target_persons', 'requirements', 'procedure', 'restrictions', 'exceptions', 'fees', 'duration_or_validity', 'quota_or_limit', 'score_criteria', 'table_rows', 'normalized_text']
fill_table = pd.DataFrame({
    '체류민원_fill_rate_%': [(stay[c].str.strip() != '').mean() * 100 for c in analysis_fields],
    '사증민원_fill_rate_%': [(visa[c].str.strip() != '').mean() * 100 for c in analysis_fields],
}, index=analysis_fields).round(1)
fill_table

## 8. 검토용 샘플 보기

필수서류, 대상, 요건 row를 일부 확인합니다.

In [ ]:
sample_cols = ['manual', 'code', 'item_type', 'subsection_type', 'petition_type', 'section_title', 'mandatory_documents', 'eligibility', 'requirements', 'normalized_text']
combined.loc[combined['subsection_type'].isin(['제출서류', '대상', '요건']), sample_cols].head(30)

## 9. 차트 파일 저장

In [ ]:
# 위 대시보드를 파일로 저장하고 싶을 때 이 셀을 실행하세요.
# 노트북에 표시된 figure와 별도로 재생성하여 저장합니다.

output_path = FIG_DIR / 'semantic_csv_overview.png'
print('저장 위치:', output_path)
# 대시보드 셀을 실행한 뒤 아래 한 줄을 별도 figure 저장 코드로 바꿔도 됩니다.
# fig.savefig(output_path, dpi=180, bbox_inches='tight')